# Assignment 4 · Milestone 4 — System Prototype
**ISBA 2411 · Due Week 7**

Deliver a working **end-to-end** prototype: takes real input → produces output → reports a preliminary evaluation against your baseline. Deliverable: GitHub repo link + 2-page technical summary.

**Reading connections**

| Pattern | Where to read |
|---|---|
| Retrieval-augmented generation (RAG) | HOLLM Ch. 8; Tunstall Ch. 7 |
| Advanced generation / tools | HOLLM Ch. 7 |
| QA / extraction pipelines | Tunstall Ch. 7; J&M Ch. 11 |

> **RAG is the default path** and a natural fit for Q&A-style projects. A classification, extraction, or summarization pipeline is **equally acceptable** — pick what fits your problem.

> **How this notebook works.** It runs **end-to-end on a small built-in demo dataset** so you can see every step working before adapting it. Find the cell marked **`# ====== SWAP IN YOUR TEAM'S DATA HERE ======`** and replace the demo data with your own — everything downstream is written to work off the same variables.

> **Compute note.** The demo uses small models (`flan-t5-small`, `distilbert-base-uncased`, `all-MiniLM-L6-v2`) that run on Colab CPU; a GPU runtime is faster for the fine-tuning step. A fixed seed is set in setup.

### How this is graded
Scored on the shared milestone rubric ([`docs/ISBA2411_Assignment_Rubric.pdf`](../docs/ISBA2411_Assignment_Rubric.pdf)):

| Rubric criterion | In this milestone |
|---|---|
| **Execution** | A working input→output pipeline with retrieval + grounded generation |
| **Analysis & Insight** | Hallucination/grounding analysis and evaluation vs. baseline |
| **Communication** | A clear architecture story for the 2-page summary |


In [ ]:
# ===== Setup (Colab) =====
!pip install -q sentence-transformers faiss-cpu transformers pandas

import numpy as np, pandas as pd, time, random
SEED = 42; random.seed(SEED); np.random.seed(SEED)

## 0. Corpus + evaluation questions
The demo corpus is a small set of policy snippets for a fictional company; the eval set is a few questions with an expected keyword in the answer.

**To use your own data:** replace `corpus` with your documents (a list of strings, or chunks of longer docs) and `qa` with your evaluation questions.

In [ ]:
# ====== SWAP IN YOUR TEAM'S DATA HERE ======
corpus = [
  'Standard orders are delivered in 5 to 7 business days after dispatch.',
  'Express shipping costs 15 dollars and delivers within 2 business days.',
  'Customers may return any item within 30 days of delivery for a full refund.',
  'Refunds are processed to the original payment method within 10 business days.',
  'The annual subscription costs 99 dollars and can be cancelled anytime.',
  'Cancelling the subscription stops future charges but does not refund the current period.',
  'Support is available by chat and email from 9am to 6pm Pacific on weekdays.',
  'Two-factor authentication can be enabled in the account security settings.',
  'Data is encrypted in transit and at rest, and backups run nightly.',
  'Enterprise plans include a dedicated account manager and a 99.9 percent uptime SLA.',
]
qa = [
  ('How long does standard shipping take?', '5 to 7'),
  ('How much does express shipping cost?', '15'),
  ('What is the return window?', '30 days'),
  ('How much is the annual subscription?', '99'),
  ('When is support available?', '9am to 6pm'),
]
print(len(corpus), 'documents |', len(qa), 'eval questions')

## 1. Inputs & outputs (the contract)
Define exactly what the system takes and returns before building it.

In [ ]:
# I/O contract for the demo system:
#   input : a natural-language question (str)
#   output: dict { 'answer': str, 'sources': list[int] (indices into corpus) }
# For your project, document your real input (a document? a record?) and output (label? summary?).
from typing import TypedDict
class RAGResult(TypedDict):
    answer: str
    sources: list  # indices of supporting documents
print('I/O contract defined')

## 2. RAG pipeline
**corpus → embed + index (FAISS) → retrieve → generate grounded answer → cite sources.**

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss, torch

embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_emb = embedder.encode(corpus, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(doc_emb.shape[1])   # inner product on normalized = cosine
index.add(doc_emb)

flan_tok = AutoTokenizer.from_pretrained('google/flan-t5-small')
flan = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-small')

def flan_generate(prompt, max_new_tokens=40):
    ids = flan_tok(prompt, return_tensors='pt').input_ids
    return flan_tok.decode(flan.generate(ids, max_new_tokens=max_new_tokens)[0], skip_special_tokens=True).strip()

def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, idx = index.search(q, k)
    return idx[0].tolist()

def rag_answer(query, k=3) -> RAGResult:
    hits = retrieve(query, k)                      # citations come from retrieval, not the model
    context = ' '.join(corpus[i] for i in hits)
    prompt = f'Answer the question based on the context below.\n\nContext: {context}\n\nQuestion: {query}'
    ans = flan_generate(prompt, max_new_tokens=40)
    return {'answer': ans, 'sources': hits}

demo = rag_answer('How long does standard shipping take?')
print('Answer :', demo['answer'])
print('Sources:', demo['sources'], '->', corpus[demo['sources'][0]])

## 3. Grounding / hallucination check
Compare the RAG answer (grounded in retrieved context) against a **no-context baseline** (the same model answering from memory). The contrast is the core argument for retrieval.

In [ ]:
def no_context_answer(query):
    return flan_generate(f'Question: {query}\nAnswer:', max_new_tokens=40)

def grounded(answer, sources):
    # crude groundedness: share of answer words that appear in the cited context
    ctx = ' '.join(corpus[i] for i in sources).lower()
    words = [w for w in answer.lower().split() if len(w) > 3]
    if not words: return 0.0
    return round(sum(w in ctx for w in words) / len(words), 2)

q = 'What is the return window?'
r = rag_answer(q)
print('RAG answer        :', r['answer'], '| groundedness =', grounded(r['answer'], r['sources']))
print('No-context answer :', no_context_answer(q))
print('\nFor your writeup: where does the no-context model hallucinate, and does retrieval fix it?')

## 4. Preliminary evaluation vs. baseline
Score the RAG pipeline against the no-retrieval baseline on the eval questions. Here the metric is **answer accuracy** (does the expected keyword appear?); use whatever metric ties back to your Milestone 2 baseline.

In [ ]:
rows = []
for question, expected in qa:
    rag = rag_answer(question)['answer']
    base = no_context_answer(question)
    rows.append(dict(question=question, expected=expected,
                     rag_correct=expected.lower() in rag.lower(),
                     baseline_correct=expected.lower() in base.lower()))
ev = pd.DataFrame(rows)
print(ev.to_string(index=False))
print(f"\nRAG accuracy      : {ev['rag_correct'].mean():.0%}")
print(f"Baseline accuracy : {ev['baseline_correct'].mean():.0%}")

## 5. Notes for the 2-page technical summary
- **Architecture diagram** — corpus → embed/index → retrieve → generate → cite.
- **Model & design choices** — embedder, generator, chunking, `k`; why each.
- **What works / what's next** — failure cases, latency/cost, and the path to Milestone 5.
- **Evaluation** — your metric, RAG vs. baseline, and how it compares to your M2 baseline.